# Notebook 03: Tokenization

**Time:** 25 minutes  
**Prerequisites:** Notebook 02 complete  
**Goal:** Understand how raw text becomes tokens — the *true* input to every LLM

This notebook will:
1. Compare three tokenizers on the same text
2. Deep-dive into BPE on diverse sentence types
3. Show how unusual tokenization leads to hallucination
4. Calculate cost implications for your project

> 💡 **Why does this matter?** The number of tokens determines your API cost, context window usage, and — surprisingly — the quality of the model's reasoning about unusual words.

In [1]:
import os, sys, json, time
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection, estimate_cost
from src.tokenizer_utils import (
    compare_tokenizers, get_tiktoken_encoder,
    tokenize_hf, estimate_tokens_tiktoken
)
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 03")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✅ Setup complete — ready for Notebook 03


---

## Part 1: Tokenizer Comparison

Three major tokenization systems are in use today:

| Tokenizer | Algorithm | Used by |
|---|---|---|
| **tiktoken cl100k_base** | BPE | GPT-4, GPT-3.5, Claude (approximation) |
| **HuggingFace gpt2** | BPE | GPT-2, many open models |
| **SentencePiece (T5)** | Unigram LM | T5, LLaMA, Mistral, multilingual models |

They all convert text → integer IDs, but with different vocabulary sizes and split points.

In [2]:
print("=" * 65)
print("🧪 Experiment 1: Side-by-Side Tokenizer Comparison")
print("=" * 65)
print()

demo_text = (
    "Large language models learn to predict the next token in a sequence. "
    "This simple objective, applied at massive scale, gives rise to remarkable emergent capabilities."
)

results = compare_tokenizers(
    text=demo_text,
    save_path=os.path.join(outputs_dir, 'tokenizer_comparison.json')
)

print()
print("💡 Notice how the same text produces different token counts")
print("   across tokenizers — this directly affects API cost!")

🧪 Experiment 1: Side-by-Side Tokenizer Comparison



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

🔤 TOKENIZER COMPARISON
Input text (165 chars):
  "Large language models learn to predict the next token in a sequence. This simple..."

Tokenizer                      Tokens   chars/token  Description
-----------------------------------------------------------------
  tiktoken_cl100k                  30           5.5  OpenAI cl100k_base (
  hf_gpt2                          30           5.5  HuggingFace gpt2 BPE
  sentencepiece_t5                 33           5.0  SentencePiece Unigra

✓ Results saved to: ../outputs/tokenizer_comparison.json

💡 Notice how the same text produces different token counts
   across tokenizers — this directly affects API cost!


### 🎯 TODO 1: Your Own Text Comparison

**Task:** Run two tokenizers on a text *you choose* and build a comparison table.

In [3]:
# TODO 1: Compare tokenizers on your own text

my_text = "[YOUR TEXT HERE — try something relevant to your project domain]"
# Examples:
#   A medical note: "Patient presents with acute myocardial infarction and hypertension..."
#   Code: "def transformer_block(x, d_model=512, num_heads=8):"
#   Multilingual: "Hello, こんにちは, مرحبا, 你好, Bonjour"

print("=" * 65)
print("🎯 TODO 1: Tokenizing my text")
print("=" * 65)

my_results = compare_tokenizers(my_text, save_path=None)  # don't overwrite the demo results

# Build a comparison table
print()
print("My comparison table:")
print()
print(f"| Tokenizer | Tokens | chars/token |")
print(f"|---|---|---|")
for name, r in my_results['results'].items():
    if 'token_count' in r:
        ratio = my_results.get('compression_ratio', {}).get(name, '?')
        print(f"| {name} | {r['token_count']} | {ratio} |")

todo1_reflection = """
[YOUR REFLECTION HERE]

- What text did you use? Why did you choose it?
- Which tokenizer was most efficient (fewest tokens)?
- Were you surprised by any splits? (Use the first_10_ids to inspect)
"""
print()
print(todo1_reflection)

🎯 TODO 1: Tokenizing my text
🔤 TOKENIZER COMPARISON
Input text (64 chars):
  "[YOUR TEXT HERE — try something relevant to your project domain]"

Tokenizer                      Tokens   chars/token  Description
-----------------------------------------------------------------
  tiktoken_cl100k                  13          4.92  OpenAI cl100k_base (
  hf_gpt2                          14          4.57  HuggingFace gpt2 BPE
  sentencepiece_t5                 19          3.37  SentencePiece Unigra


My comparison table:

| Tokenizer | Tokens | chars/token |
|---|---|---|
| tiktoken_cl100k | 13 | 4.92 |
| hf_gpt2 | 14 | 4.57 |
| sentencepiece_t5 | 19 | 3.37 |


[YOUR REFLECTION HERE]

- What text did you use? Why did you choose it?
- Which tokenizer was most efficient (fewest tokens)?
- Were you surprised by any splits? (Use the first_10_ids to inspect)



---

## Part 2: BPE Deep-Dive — Diverse Sentence Types

BPE (Byte Pair Encoding) builds its vocabulary by iteratively merging the most frequent byte pairs.
The result: common English words get a single token; rare/unusual strings get split into many.

In [4]:
print("=" * 65)
print("🧪 Experiment 2: tiktoken on Diverse Sentence Types")
print("=" * 65)
print()

enc = get_tiktoken_encoder("cl100k_base")

test_sentences = [
    ("English prose",     "The transformer architecture revolutionized natural language processing."),
    ("Python code",       "def attention(Q, K, V):\n    return softmax(Q @ K.T / sqrt(d_k)) @ V"),
    ("Emoji & symbols",   "🎉 AI is amazing! 100% 😊 → ∞"),
    ("Multilingual",      "Hello こんにちは مرحبا 你好 Bonjour"),
    ("Math formula",      "E = mc² where c ≈ 2.998×10⁸ m/s"),
]

print(f"{'Type':<20} {'Tokens':>8}  {'chars/tok':>10}  First 5 tokens")
print("-" * 65)

for sentence_type, text in test_sentences:
    ids    = enc.encode(text)
    ratio  = round(len(text) / len(ids), 1) if ids else 0
    first5 = enc.decode_single_token_bytes(ids[0]).decode('utf-8', errors='replace') if ids else ''
    token_strs = [repr(enc.decode([i])) for i in ids[:5]]
    print(f"  {sentence_type:<18} {len(ids):>8}  {ratio:>10}  {token_strs}")

print()
print("💡 Key observation: English prose has the highest chars-per-token ratio")
print("   (most efficient). Emojis and non-Latin scripts often need many tokens.")

🧪 Experiment 2: tiktoken on Diverse Sentence Types

Type                   Tokens   chars/tok  First 5 tokens
-----------------------------------------------------------------
  English prose             9         8.0  ["'The'", "' transformer'", "' architecture'", "' revolution'", "'ized'"]
  Python code              22         3.0  ["'def'", "' attention'", "'(Q'", "','", "' K'"]
  Emoji & symbols          15         1.8  ["'�'", "'�'", "'�'", "' AI'", "' is'"]
  Multilingual             13         2.2  ["'Hello'", "' '", "'こんにちは'", "' م'", "'ر'"]
  Math formula             18         1.7  ["'E'", "' ='", "' mc'", "'²'", "' where'"]

💡 Key observation: English prose has the highest chars-per-token ratio
   (most efficient). Emojis and non-Latin scripts often need many tokens.


### 🎯 TODO 2: Compare with GPT-2 Tokenizer

In [5]:
# TODO 2: Repeat the experiment with HuggingFace gpt2 tokenizer and compare

print("=" * 65)
print("🎯 TODO 2: Same Sentences — GPT-2 Tokenizer")
print("=" * 65)
print()

try:
    from transformers import AutoTokenizer
    gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

    print(f"{'Type':<20} {'tiktoken':>10} {'gpt2':>8}  {'Diff':>8}")
    print("-" * 55)

    for sentence_type, text in test_sentences:
        tiktoken_count = len(enc.encode(text))
        gpt2_count     = len(gpt2_tok.encode(text))
        diff = gpt2_count - tiktoken_count
        sign = '+' if diff > 0 else ''
        print(f"  {sentence_type:<18} {tiktoken_count:>10} {gpt2_count:>8}  {sign}{diff:>7}")

    print()

    # TODO: Fill in your reflection
    todo2_reflection = """
[YOUR REFLECTION HERE]

- For which sentence types does gpt2 use more tokens than tiktoken? Why?
- cl100k_base (tiktoken) has 100K vocab vs gpt2's 50K vocab — how does that affect efficiency?
- If you're using Claude (which uses a similar vocab to cl100k), what does this mean for cost estimation?
"""
    print(todo2_reflection)

except ImportError:
    print("⚠️  transformers not installed — run: pip install transformers")

🎯 TODO 2: Same Sentences — GPT-2 Tokenizer

Type                   tiktoken     gpt2      Diff
-------------------------------------------------------
  English prose               9        9        0
  Python code                22       32  +     10
  Emoji & symbols            15       14       -1
  Multilingual               13       22  +      9
  Math formula               18       18        0


[YOUR REFLECTION HERE]

- For which sentence types does gpt2 use more tokens than tiktoken? Why?
- cl100k_base (tiktoken) has 100K vocab vs gpt2's 50K vocab — how does that affect efficiency?
- If you're using Claude (which uses a similar vocab to cl100k), what does this mean for cost estimation?



---

## Part 3: Token Boundaries and Hallucination

LLMs never see characters — they only see token IDs. When a word splits into unexpected tokens,
the model may struggle to reason about it correctly. This is one cause of hallucination.

In [7]:
print("=" * 65)
print("🧪 Experiment 3: Token Boundaries & Unusual Strings")
print("=" * 65)
print()

unusual_strings = [
    "ChatGPT",
    "claude-sonnet-4-6",
    "uncopyrightable",
    "https://arxiv.org/abs/1706.03762",
    "2024-12-31T23:59:59Z",
    "<|endoftext|>",
    "99999999",
]

print(f"{'String':<35} {'Tokens':>8}  Token pieces")
print("-" * 70)

for s in unusual_strings:
    ids    = enc.encode(s, disallowed_special=())  # allow special tokens as normal text
    pieces = [enc.decode([i]) for i in ids]
    print(f"  {s:<33} {len(ids):>8}  {pieces}")

print()
print("💡 'ChatGPT' splits into ['Chat', 'G', 'PT'] or similar.")
print("   The model has never 'seen' this as one unit — it reasons from pieces.")
print()
print("   This explains why LLMs struggle with: letter counting ('strawberry'),")
print("   anagram tasks, and unusual proper nouns.")

🧪 Experiment 3: Token Boundaries & Unusual Strings

String                                Tokens  Token pieces
----------------------------------------------------------------------
  ChatGPT                                  3  ['Chat', 'G', 'PT']
  claude-sonnet-4-6                        9  ['cla', 'ude', '-', 'son', 'net', '-', '4', '-', '6']
  uncopyrightable                          3  ['unc', 'opyright', 'able']
  https://arxiv.org/abs/1706.03762        13  ['https', '://', 'ar', 'xiv', '.org', '/', 'abs', '/', '170', '6', '.', '037', '62']
  2024-12-31T23:59:59Z                    13  ['202', '4', '-', '12', '-', '31', 'T', '23', ':', '59', ':', '59', 'Z']
  <|endoftext|>                            7  ['<', '|', 'endo', 'ft', 'ext', '|', '>']
  99999999                                 3  ['999', '999', '99']

💡 'ChatGPT' splits into ['Chat', 'G', 'PT'] or similar.
   The model has never 'seen' this as one unit — it reasons from pieces.

   This explains why LLMs struggle with: l

---

## Part 4: Cost Implications for Your Project

Every token costs money. Let's calculate the budget impact for your project.

In [8]:
print("=" * 65)
print("🧪 Experiment 4: Cost Comparison Across Models")
print("=" * 65)
print()

# 500-word paragraph (typical LLM prompt)
sample_prompt = """You are an expert assistant helping analyze research papers in the field of
natural language processing. I will provide you with a paper abstract, and you should
extract the key contributions, methodology, and results in a structured format.

Paper abstract: The transformer architecture, introduced in the seminal paper 'Attention
Is All You Need', has become the dominant paradigm in natural language processing.
This survey provides a comprehensive overview of transformer variants developed
since the original work, categorizing them by architectural changes, training
objectives, and downstream task performance. We analyze over 200 papers published
between 2017 and 2025, identifying key trends including: the shift from encoder-decoder
to decoder-only architectures, the scaling laws governing parameter-performance
relationships, the emergence of in-context learning at scale, and recent work on
mixture-of-experts models. Our analysis reveals that architectural innovations
beyond scaling have had diminishing returns, while data quality and diversity
remain critical factors for achieving state-of-the-art performance.
Please provide: (1) Main contribution, (2) Methodology summary, (3) Key findings,
(4) Limitations mentioned, (5) Your assessment of novelty."""

models = {
    "claude-sonnet-4-6":   {"input": 3.0, "output": 15.0},
    "claude-opus-4-6":     {"input": 15.0, "output": 75.0},
    "claude-haiku-4-5-20251001": {"input": 1.0, "output": 5.0},
}

input_tokens  = estimate_tokens_tiktoken(sample_prompt)
output_tokens = 300   # typical response length

print(f"Sample prompt: {len(sample_prompt)} chars, ~{input_tokens} tokens")
print(f"Expected response: ~{output_tokens} tokens")
print()
print(f"{'Model':<30} {'Cost/call':>12}  {'1000 calls':>12}")
print("-" * 58)

for model, pricing in models.items():
    cost_per_call = (
        (input_tokens / 1_000_000) * pricing['input'] +
        (output_tokens / 1_000_000) * pricing['output']
    )
    print(f"  {model:<28} ${cost_per_call:.5f}  ${cost_per_call * 1000:.2f}")

print()
print("💡 For this homework (~50 API calls): haiku ~$0.003, sonnet ~$0.10, opus ~$0.50")

🧪 Experiment 4: Cost Comparison Across Models

Sample prompt: 1269 chars, ~241 tokens
Expected response: ~300 tokens

Model                             Cost/call    1000 calls
----------------------------------------------------------
  claude-sonnet-4-6            $0.00522  $5.22
  claude-opus-4-6              $0.02611  $26.11
  claude-haiku-4-5-20251001    $0.00174  $1.74

💡 For this homework (~50 API calls): haiku ~$0.003, sonnet ~$0.10, opus ~$0.50


### 🎯 TODO 3: Your Project's Token Budget

In [9]:
# TODO 3: Run YOUR project text through the token counter and plan your budget

# Paste a representative sample of text you'd send to an LLM in your project
my_project_prompt = """
[PASTE YOUR PROJECT'S TYPICAL PROMPT HERE]

Example: If you're building a medical QA bot, paste a typical question + context.
         If you're building a code reviewer, paste a typical code snippet.
"""

my_expected_output_tokens = 500   # ← Set your expected response length

print("=" * 65)
print("🎯 TODO 3: My Project Token Budget")
print("=" * 65)
print()

my_input_tokens = estimate_tokens_tiktoken(my_project_prompt)
print(f"My prompt:")
print(f"  Chars:  {len(my_project_prompt):,}")
print(f"  Tokens: {my_input_tokens:,}  (tiktoken estimate)")
print(f"  Expected output: {my_expected_output_tokens} tokens")
print()

print(f"{'Model':<30} {'Cost/call':>12}  {'Budget for 100 calls':>22}")
print("-" * 68)
for model, pricing in models.items():
    cost = (
        (my_input_tokens / 1_000_000) * pricing['input'] +
        (my_expected_output_tokens / 1_000_000) * pricing['output']
    )
    print(f"  {model:<28} ${cost:.5f}  ${cost * 100:.3f}")

# Make an actual API call to test the prompt
print()
print("Sending a test call...")
test_response = client.generate(
    prompt=my_project_prompt[:500],   # truncate for test
    system="Reply in 2 sentences.",
    max_tokens=100,
    temperature=0.5,
)
if "error" not in test_response:
    tracker.add_call(test_response)
    print(f"✅ Actual token count: {test_response['usage']['input_tokens']}in / {test_response['usage']['output_tokens']}out")

# TODO: Write your reflection
todo3_reflection = """
[YOUR REFLECTION HERE]

- What is the estimated cost per call for your project?
- How does this affect your model choice (Haiku vs Sonnet vs Opus)?
- Can you reduce token count without losing important information?
  (e.g., by trimming context, using shorter system prompts)
- What's your total estimated API budget for the semester project?
"""
print()
print(todo3_reflection)

🎯 TODO 3: My Project Token Budget

My prompt:
  Chars:  202
  Tokens: 46  (tiktoken estimate)
  Expected output: 500 tokens

Model                             Cost/call    Budget for 100 calls
--------------------------------------------------------------------
  claude-sonnet-4-6            $0.00764  $0.764
  claude-opus-4-6              $0.03819  $3.819
  claude-haiku-4-5-20251001    $0.00255  $0.255

Sending a test call...
✅ Actual token count: 66in / 48out


[YOUR REFLECTION HERE]

- What is the estimated cost per call for your project?
- How does this affect your model choice (Haiku vs Sonnet vs Opus)?
- Can you reduce token count without losing important information?
  (e.g., by trimming context, using shorter system prompts)
- What's your total estimated API budget for the semester project?



## Summary & Reflection

In [10]:
full_reflection = f"""
### Tokenizer Comparison

Demo text token counts:
- tiktoken cl100k_base: {results['results'].get('tiktoken_cl100k', {}).get('token_count', 'N/A')} tokens
- HF gpt2: {results['results'].get('hf_gpt2', {}).get('token_count', 'N/A')} tokens
- SentencePiece T5: {results['results'].get('sentencepiece_t5', {}).get('token_count', 'N/A')} tokens

**My text comparison (TODO 1):**
{todo1_reflection.strip()}

---

### BPE Deep-Dive (TODO 2)

{todo2_reflection.strip() if 'todo2_reflection' in dir() else 'See cell output above.'}

---

### Token Budget for My Project (TODO 3)

My prompt: {my_input_tokens} tokens
Expected output: {my_expected_output_tokens} tokens

{todo3_reflection.strip()}
"""

reflection_file = append_to_reflection(
    notebook="03",
    section_title="Tokenization & Cost Analysis",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"✅ Reflection saved: {reflection_file}")
print(f"✅ tokenizer_comparison.json saved to: {outputs_dir}")
print()
tracker.report()

✅ Reflection saved: ../outputs/homework_reflection.md
✅ tokenizer_comparison.json saved to: ../outputs

💰 API COST REPORT
Total API calls:     1
Total input tokens:  66
Total output tokens: 48
Total cost:          $0.0009

Last 1 calls:
  1. [22:56:45] sonnet — 66in/48out — $0.0009


## ✅ Notebook 03 Complete!

**What you accomplished:**
- ✅ Compared three major tokenizers side-by-side
- ✅ Understood BPE behaviour across English, code, emoji, multilingual, and math text
- ✅ Saw how unusual token boundaries contribute to hallucination
- ✅ Calculated real cost implications for your project
- ✅ Saved `outputs/tokenizer_comparison.json`

**Key concepts:**
- BPE builds vocabulary by merging frequent byte pairs → common words = 1 token
- Larger vocabulary = fewer tokens per text = lower cost
- LLMs reason about *tokens*, not characters — unusual splits cause reasoning errors
- Always use tiktoken (not char/4 heuristic) for accurate cost estimates

**Next:** Open **Notebook 04: Pretraining Data Collection** 🌐